# CRISP-DM Fase 6: Deployment & Communication - Visualizaciones Ejecutivas
## Dataset: SIPSA Precios Mayoristas (DANE)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 6: Deployment & Business Insights) | **Fase PDCO**: OPERATIONS  
**Estándares Aplicados**: Data Storytelling, Cartas de Control Estadístico de Shewhart/Nelson, Toma de Decisiones Estratégicas.

---

### 1. El Rol de la Visualización en el Despliegue Analítico (CRISP-DM Fase 6)
La última etapa de CRISP-DM no consiste únicamente en exportar modelos, sino en **comunicar hallazgos de forma accionable y visualmente comprensible** a los directores gremiales, agricultores y formuladores de política pública.
* **Cartas de Control Estadístico**: Representan la serie temporal junto con la media ($\\mu$) y los límites de control superior e inferior ($\\mu \\pm 3\\sigma$), facilitando la detección visual inmediata de anomalías agroclimáticas o de mercado.
* **Bandas de Confianza del 95%**: Ilustran el rango esperado de variación bajo condiciones normales de operación.
* **Cuadros de Mando Ejecutivos**: Sintetizan los KPIs para dar respuesta clara a las preguntas de la Batería A1-J1.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# Importación estándar y resolución dinámica del path para src/
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


### 2. Generación de Gráficos de Control y Comparativa Territorial
Generamos un panel visual dual con la serie cronológica bajo límites de control estadístico y la distribución espacial/categórica por nodos principales.


In [ ]:
# ==============================================================================
# [FASE 8: GENERACIÓN DE GRÁFICOS EJECUTIVOS PARA TOMA DE DECISIONES]
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
from src.database.db_manager import DatabaseManager

db = DatabaseManager(str(APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"))
df = db.query("SELECT * FROM sipsa_precios")

primary_col = "precio_promedio"
group_col = "mercado"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if primary_col in df.columns and pd.api.types.is_numeric_dtype(df[primary_col]):
    # 1. Gráfico de Control y Tendencia (Series)
    s = df[primary_col].dropna().reset_index(drop=True)
    mu, sigma = s.mean(), s.std()
    
    axes[0].plot(s.index[:100], s.values[:100], color="#1d3557", lw=1.5, label="Observaciones")
    axes[0].axhline(mu, color="#2a9d8f", linestyle="-", label=f"Media: {mu:,.2f}")
    axes[0].axhline(mu + 3*sigma, color="#e63946", linestyle="--", label="Límite Superior (+3σ)")
    axes[0].axhline(max(0, mu - 3*sigma), color="#e63946", linestyle="--", label="Límite Inferior (-3σ)")
    axes[0].fill_between(s.index[:100], max(0, mu - 2*sigma), mu + 2*sigma, color="#a8dadc", alpha=0.3, label="Zona Control 95%")
    axes[0].set_title(f"Carta de Control Estadístico: {primary_col} (SIPSA Precios Mayoristas (DANE))", fontsize=11, fontweight="bold")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_ylabel(primary_col)
    
    # 2. Gráfico por Categorías / Nodos Territoriales
    if group_col in df.columns:
        top_cats = df.groupby(group_col)[primary_col].median().sort_values(ascending=False).head(8)
        sns.barplot(x=top_cats.values, y=top_cats.index, ax=axes[1], palette="crest")
        axes[1].set_title(f"Mediana por {group_col} (Top 8 Nodos)", fontsize=11, fontweight="bold")
        axes[1].set_xlabel("Mediana " + primary_col)
else:
    axes[0].text(0.5, 0.5, "Dataset Estructurado Documental", ha="center", va="center")
    axes[1].text(0.5, 0.5, "Revisión de Frecuencias Textuales", ha="center", va="center")

plt.tight_layout()
plt.show()


### 3. Cuadro de Mando de KPIs y Recomendaciones de Operación
Consolidamos las conclusiones operativas para la toma de decisiones basada en datos.


In [ ]:
# ==============================================================================
# [FASE 8: CUADRO DE MANDO Y RECOMENDACIONES DE NEGOCIO (CRISP-DM)]
# ==============================================================================
print("=" * 70)
print(f"🌾 SÍNTESIS EJECUTIVA DE NEGOCIO: SIPSA Precios Mayoristas (DANE)")
print("=" * 70)
print(f"• Objetivo Estratégico: Identificar patrones de volatilidad, shocks de precios y tendencias estacionales en plazas de mercado mayoristas para la toma de decisiones comerciales y mitigación de riesgo.")
print(f"• Entidad Analítica: Cotizaciones Mayoristas Diarias de Productos Agrícolas (COP/Kg)")
print(f"• Custodio Oficial: DANE - Sistema de Información de Precios y Abastecimiento del Sector Agropecuario (SIPSA)")
print("\n💡 RECOMENDACIONES ACCIONABLES PARA TOMA DE DECISIONES:")
print("1. Cobertura Financiera: Monitorear alertas de volatilidad para estructurar coberturas y compras anticipadas.")
print("2. Articulación Logística: Optimizar rutas de transporte entre municipios emisores y centrales mayoristas.")
print("3. Alertas Preventivas: Configurar notificaciones automáticas ante superación de los límites de control de 3-sigma.")
print("=" * 70)
print("✅ CICLO DE VIDA COMPLETO CRISP-DM FINALIZADO EXITOSAMENTE.")


### 4. Cierre del Ciclo de Vida CRISP-DM
El ciclo completo desde la Comprensión del Negocio (Fase 1) hasta el Despliegue de Resultados (Fase 8) ha sido ejecutado con rigor científico, validación de calidad y trazabilidad formal.
